In [1]:
#Importamos el csv con la info:
path_origen = r'C:\Users\pedro\.cache\kagglehub\datasets\ramanathansp20\inbreast-dataset\versions\1\INbreast Release 1.0'

import pandas as pd
import os
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

INbreast = pd.read_csv(os.path.join(path_origen, 'INbreast.csv'), sep = ';')
INbreast

,Patient ID,Patient age,Laterality,View,Acquisition date,File Name,ACR,Bi-Rads
0,removed,removed,R,CC,201001,22678622,4,1
1,removed,removed,L,CC,201001,22678646,4,3
2,removed,removed,R,MLO,201001,22678670,4,1
3,removed,removed,L,MLO,201001,22678694,4,3
4,removed,removed,R,CC,201001,22614074,2,5
5,removed,removed,L,CC,201001,22614097,2,2
6,removed,removed,R,MLO,201001,22614127,2,5
7,removed,removed,L,MLO,201001,22614150,2,2
8,removed,removed,L,MLO,201001,50997434,3,2
9,removed,removed,R,MLO,201001,50997461,3,4a


In [2]:
#Convertimos DICOM a PNG

path_destino = r'C:\Users\pedro\Documents\MÁSTER UNIR 💻\TFM\extracción de imágenes del dataset\Imagenes png INbreast'
import pydicom
import numpy as np
from PIL import Image

def dicom_to_png(path_dcm, path_png):
    ds = pydicom.dcmread(path_dcm)
    img = ds.pixel_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min())
    img = (img * 255).astype(np.uint8)
    Image.fromarray(img).save(path_png)



#Hay que hacer un bucle para todas las imágenes
#Primero necesitamos los nombres de las imágenes de la carpeta:
carpeta_dcms = os.path.join(path_origen, 'AllDICOMs')
lista_archivos = os.listdir(carpeta_dcms)

for e in lista_archivos:   
    path_origen_im = os.path.join(carpeta_dcms, str(e))
    e = e.replace('.dcm', '')
    path_destino_im = os.path.join(path_destino, str(e)+'.png')
    
    dicom_to_png(path_origen_im, path_destino_im)


In [15]:
#Parsear las máscaras del tumor para convertirlas en imágenes binarias

import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image, ImageDraw

#Primero definimos la función para pasar se .xml a un array
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image, ImageDraw
import re

def parse_inbreast_xml(xml_path, image_shape, radius_calcif=5):
    """
    xml_path: ruta al XML de INbreast (formato plist)
    image_shape: (height, width) de la imagen DICOM
    radius_calcif: radio en píxeles para marcar calcificaciones puntuales
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # máscara vacía
    mask = Image.new("L", (image_shape[1], image_shape[0]), 0)
    draw = ImageDraw.Draw(mask)

    # buscamos el array asociado a la key "ROIs"
    rois_array = None
    for i, elem in enumerate(root.iter()):
        if elem.tag == "key" and elem.text == "ROIs":
            # el siguiente elemento en el árbol debe ser el <array> de ROIs
            # ojo: root.iter() es un generador, así que mejor recorrer root y usar índice
            parent = elem.getparent() if hasattr(elem, "getparent") else None
    # Como ElementTree estándar no tiene getparent, hacemos un recorrido manual:

    # Versión robusta: recorrer todos los elementos y guardar en lista
    all_elems = list(root.iter())
    rois_array = None
    for idx, elem in enumerate(all_elems):
        if elem.tag == "key" and elem.text == "ROIs":
            # el siguiente elemento debe ser el <array>
            if idx + 1 < len(all_elems) and all_elems[idx + 1].tag == "array":
                rois_array = all_elems[idx + 1]
            break

    if rois_array is None:
        # No hay ROIs → máscara vacía
        return np.array(mask)

    # cada <dict> dentro de rois_array es una ROI
    for roi_dict in rois_array.findall("dict"):
        points_px = []

        children = list(roi_dict)
        for i, elem in enumerate(children):
            if elem.tag == "key" and elem.text == "Point_px":
                arr = children[i + 1]  # debería ser <array>
                for s in arr.findall("string"):
                    # s.text = "(2964.620117, 3427.979980)" o "(x, y, z)"
                    nums = re.findall(r"[-+]?\d*\.\d+|\d+", s.text)
                    if len(nums) >= 2:
                        x = float(nums[0])
                        y = float(nums[1])
                        points_px.append((x, y))

        if len(points_px) == 0:
            continue

        if len(points_px) == 1:
            # calcificación puntual → dibujamos un pequeño círculo
            x, y = points_px[0]
            x0 = x - radius_calcif
            y0 = y - radius_calcif
            x1 = x + radius_calcif
            y1 = y + radius_calcif
            draw.ellipse([x0, y0, x1, y1], outline=1, fill=1)
        else:
            # masa / contorno → polígono
            draw.polygon(points_px, outline=1, fill=1)

    return np.array(mask)

In [18]:
#ahora hacemos un bucle para convertir en png y guardar
num_pixeles_mascara = []

import pydicom
carpeta_dcms = os.path.join(path_origen, 'AllDICOMs')
lista_archivos = os.listdir(carpeta_dcms)

for e in lista_archivos: 
    #obtenemos la altura y anchura de las fotos en dicom
    path_origen_im = os.path.join(carpeta_dcms, str(e))
    
    try:
        ds = pydicom.dcmread(path_origen_im)
    except pydicom.errors.InvalidDicomError:
        print("Archivo no válido DICOM, se salta:", e)
        continue
    img = ds.pixel_array
    image_shape = img.shape

    #Obtenemos el path de cada .xml
    carpeta_xml = os.path.join(path_origen, 'AllXML')
    partes = e.split('_')
    path_xml = os.path.join(carpeta_xml, partes[0] + '.xml')

    #Si no existe el .xml para ese dicom, nos lo saltamos
    if not os.path.exists(path_xml):
        print("No hay XML para:", e)
        continue

    #Aplicamos la función
    
    mask_array = parse_inbreast_xml(path_xml, image_shape)
    mask_array_byn = (mask_array * 255).astype(np.uint8)

    #comprobamos que las máscaras que están saliendo tienen algún píxel no negro
    num_pixeles_mascara.append(mask_array.sum())
    #Decidimos el nombre de la imagen:
    nombre_mascara = e.replace('.dcm','_mask.png')
    #guardamos imagen
    Image.fromarray(mask_array_byn).save(
        os.path.join('Imagenes mascaras segmentacion INbreast', nombre_mascara))
    

No hay XML para: 20588138_8d0b9620c53c0268_MG_R_ML_ANON.dcm
No hay XML para: 20588164_8d0b9620c53c0268_MG_R_CC_ANON.dcm
No hay XML para: 22580218_5530d5782fc89dd7_MG_L_CC_ANON.dcm
No hay XML para: 22580270_5530d5782fc89dd7_MG_L_ML_ANON.dcm
No hay XML para: 22613848_45c7f44839fd9e68_MG_L_ML_ANON.dcm
No hay XML para: 22613944_f23fa352e7de3dc7_MG_L_CC_ANON.dcm
No hay XML para: 22613996_f23fa352e7de3dc7_MG_L_ML_ANON.dcm
No hay XML para: 22670442_7e677f3d530e41ed_MG_R_CC_ANON.dcm
No hay XML para: 22670488_7e677f3d530e41ed_MG_R_ML_ANON.dcm
No hay XML para: 22670643_e15a16f87b4f9782_MG_L_CC_ANON.dcm
No hay XML para: 22670703_e15a16f87b4f9782_MG_L_ML_ANON.dcm
No hay XML para: 22678622_61b13c59bcba149e_MG_R_CC_ANON.dcm
No hay XML para: 22678670_61b13c59bcba149e_MG_R_ML_ANON.dcm
No hay XML para: 24054997_2f1104b3cda7f145_MG_L_ML_ANON.dcm
No hay XML para: 24055051_2f1104b3cda7f145_MG_L_CC_ANON.dcm
No hay XML para: 24065270_c4b995eddb3c510c_MG_R_ML_ANON.dcm
No hay XML para: 24065308_c4b995eddb3c51

In [17]:
#Comprobamos si las máscaras tiene algún píxel no negro
import pandas as pd
serie = pd.Series(num_pixeles_mascara)
print(serie.value_counts())

388       4
485       4
194       4
582       3
97        3
457       2
776       2
1321      2
874       2
676       2
1122      2
4557      1
37548     1
65614     1
763       1
8557      1
16787     1
53        1
52        1
375       1
405       1
474659    1
466697    1
4330      1
61745     1
3992      1
50457     1
18722     1
535589    1
8291      1
16520     1
164       1
11112     1
66        1
307647    1
383783    1
173914    1
1492      1
113205    1
25577     1
22645     1
101044    1
79917     1
291       1
284901    1
256570    1
238       1
446180    1
1066      1
34011     1
986       1
26799     1
141421    1
989       1
132864    1
618       1
1040      1
27246     1
23813     1
4684      1
82794     1
5049      1
84715     1
31407     1
19889     1
307339    1
198596    1
208216    1
155364    1
1455      1
266996    1
2485      1
344757    1
752487    1
8084      1
490576    1
12436     1
233573    1
142       1
196123    1
718752    1
122224    1
4485      1
3613

In [3]:
#Extraemos ROI (sub-foto de la mínima región que abarca toda la máscara binaria segmentada)
import numpy as np
from PIL import Image
import os

def extract_roi(img, mask, scale=1.2):
    ys, xs = np.where(mask == 1)
    if len(xs) == 0 or len(ys) == 0:
        return None

    ymin, ymax = ys.min(), ys.max()
    xmin, xmax = xs.min(), xs.max()

    h = ymax - ymin
    w = xmax - xmin

    cy = (ymin + ymax) // 2
    cx = (xmin + xmax) // 2

    new_h = int(h * scale)
    new_w = int(w * scale)

    ymin_new = max(cy - new_h // 2, 0)
    ymax_new = min(cy + new_h // 2, img.shape[0])
    xmin_new = max(cx - new_w // 2, 0)
    xmax_new = min(cx + new_w // 2, img.shape[1])

    roi = img[ymin_new:ymax_new, xmin_new:xmax_new]
    return roi

#Ahora con un bucle generamos y guardamos las ROI
lista_archivos = os.listdir('Imagenes mascaras segmentacion INbreast RENOMBRADAS')
for e in lista_archivos:
    nombre_imagen = e.replace('_mask', '')
    ruta_imagen = os.path.join('Imagenes png INbreast RENOMBRADAS', nombre_imagen) 
    ruta_mask =  os.path.join('Imagenes mascaras segmentacion INbreast RENOMBRADAS', e)
    
    img = np.array(Image.open(ruta_imagen))
    mask = np.array(Image.open(ruta_mask)) // 255  # convertir a 0/1
                    
    roi_img = extract_roi(img, mask)
    if roi_img is not None:
        #Nombre del roi y guardamos:
        nombre_roi = e.replace('_mask','_roi')
        Image.fromarray(roi_img).save(os.path.join('Imagenes ROI INbreast RENOMBRADAS CON MARGEN', nombre_roi))


In [4]:
#ROI ESPECÍFICO PARA IMÁGENES MASS_MICRO: QUEREMOS QUE SOLO SE CENTRE EN LA MASA E IGNORE LOS MICRO
import numpy as np
from scipy.ndimage import label

def extract_roi_onlybiggest(img, mask, scale=1.2):
    # 1. Etiquetar componentes conectados
    labeled, num = label(mask)

    if num == 0:
        return None  # no hay lesión

    # 2. Calcular el área de cada componente
    areas = [(labeled == i).sum() for i in range(1, num+1)]

    # 3. Seleccionar el componente más grande (la MASS)
    largest_label = np.argmax(areas) + 1
    mask_mass = (labeled == largest_label)

    # 4. Bounding box SOLO de la MASS
    ys, xs = np.where(mask_mass)
    ymin, ymax = ys.min(), ys.max()
    xmin, xmax = xs.min(), xs.max()

    # 5. Expandir ROI con factor scale
    h = ymax - ymin
    w = xmax - xmin

    cy = (ymin + ymax) // 2
    cx = (xmin + xmax) // 2

    new_h = int(h * scale)
    new_w = int(w * scale)

    ymin_new = max(cy - new_h // 2, 0)
    ymax_new = min(cy + new_h // 2, img.shape[0])
    xmin_new = max(cx - new_w // 2, 0)
    xmax_new = min(cx + new_w // 2, img.shape[1])

    # 6. Extraer ROI centrado en la MASS
    roi = img[ymin_new:ymax_new, xmin_new:xmax_new]
    return roi

#Ahora hacemos un bucle que cree este ROI SOLO para imágenes con MASS_MICRO
lista_archivos = os.listdir('Imagenes mascaras segmentacion INbreast RENOMBRADAS')
lista_archivos = [x for x in lista_archivos if x.startswith('MASS_MICRO')]

for e in lista_archivos:
    nombre_imagen = e.replace('_mask', '')
    ruta_imagen = os.path.join('Imagenes png INbreast RENOMBRADAS', nombre_imagen) 
    ruta_mask =  os.path.join('Imagenes mascaras segmentacion INbreast RENOMBRADAS', e)
    
    img = np.array(Image.open(ruta_imagen))
    mask = np.array(Image.open(ruta_mask)) // 255  # convertir a 0/1
                    
    roi_img = extract_roi_onlybiggest(img, mask)
    if roi_img is not None:
        #Nombre del roi y guardamos:
        nombre_roi = e.replace('_mask','_roi')
        Image.fromarray(roi_img).save(os.path.join('Imagenes ROI INbreast RENOMBRADAS CON MARGEN', nombre_roi))
